# Análisis de Vulnerabilidades y SAST

Este notebook consolida y analiza los resultados de:
- **Grype**: vulnerabilidades de dependencias (desde SBOMs generados por Syft)
- **Semgrep**: hallazgos de análisis estático de código fuente (SAST)

Los datos se cargan desde los directorios generados por `scripts/generate_sboms.py`.
Para reproducir el análisis, ejecute primero el pipeline completo y luego ejecute todas las celdas en orden.

### Configuración inicial

Se importan las librerías necesarias y se definen los parámetros configurables.
Modifique las rutas en `VULNS_DIR`, `SAST_DIR` y `SBOMS_DIR` si los resultados están en otra ubicación.

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual
sns.set_theme(style="whitegrid")

# ============================================================
# PARÁMETROS CONFIGURABLES
# ============================================================
VULNS_DIR = "../../data/results/vulns"
SAST_DIR = "../../data/results/sast"
SBOMS_DIR = "../../data/results/sboms"
SEVERITY_ORDER = ["Critical", "High", "Medium", "Low", "Negligible", "Unknown"]

---
## 1. Vulnerabilidades de Dependencias (Grype)

Grype escanea los SBOMs generados por Syft y produce un JSON con la estructura:
```
{ "matches": [ { "vulnerability": {...}, "artifact": {...} } ] }
```
Cada match contiene la información de la vulnerabilidad (ID, severidad) y del paquete afectado (nombre, versión).

**Carga y parseo de resultados Grype:** Se iteran todos los archivos JSON en `VULNS_DIR`, se extraen los campos relevantes de cada match y se construye un DataFrame unificado.

In [ ]:
all_vulns = []

for filename in os.listdir(VULNS_DIR):
    if filename.endswith(".json"):
        filepath = os.path.join(VULNS_DIR, filename)
        with open(filepath, "r") as f:
            data = json.load(f)
            repo_name = filename.replace("_vuln.json", "")

            matches = data.get("matches", [])
            for match in matches:
                vuln = match.get("vulnerability", {})
                artifact = match.get("artifact", {})

                all_vulns.append({
                    "Repository": repo_name,
                    "Vulnerability ID": vuln.get("id"),
                    "Severity": vuln.get("severity"),
                    "Package": artifact.get("name"),
                    "Version": artifact.get("version")
                })

df_vulns = pd.DataFrame(all_vulns)
print(f"Total de vulnerabilidades de dependencias: {len(df_vulns)}")
if not df_vulns.empty:
    display(df_vulns.head())

**Distribución por severidad:** Gráfico de barras que muestra cuántas vulnerabilidades hay en cada nivel (Critical > High > Medium > Low). Permite identificar rápidamente la proporción de hallazgos críticos.

In [ ]:
if not df_vulns.empty:
    plt.figure(figsize=(10, 6))
    ax = sns.countplot(
        data=df_vulns,
        x="Severity",
        hue="Severity",
        order=[s for s in SEVERITY_ORDER if s in df_vulns["Severity"].values],
        palette="Reds_r",
        legend=False
    )
    plt.title("Distribución de Vulnerabilidades por Severidad (Grype)", fontsize=14)
    plt.ylabel("Cantidad")
    plt.xlabel("Nivel de Severidad")

    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=10, color='black', xytext=(0, 5),
                    textcoords='offset points')
    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron vulnerabilidades de dependencias.")

**Vulnerabilidades por repositorio:** Muestra la distribución de hallazgos por cada repo, desglosada por severidad. Útil para identificar qué repositorios concentran más riesgo.

In [ ]:
if not df_vulns.empty:
    plt.figure(figsize=(12, 6))
    ax = sns.countplot(
        data=df_vulns,
        x="Repository",
        hue="Severity",
        hue_order=[s for s in SEVERITY_ORDER if s in df_vulns["Severity"].values],
        palette="Reds_r"
    )
    plt.title("Vulnerabilidades por Repositorio y Severidad", fontsize=14)
    plt.ylabel("Cantidad")
    plt.xlabel("Repositorio")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("Sin datos para graficar por repositorio.")

**Top 10 paquetes más vulnerables:** Identifica las dependencias que aparecen con más vulnerabilidades. Estos son los candidatos prioritarios para actualización o reemplazo.

In [ ]:
if not df_vulns.empty:
    top_pkgs = df_vulns["Package"].value_counts().head(10)
    plt.figure(figsize=(10, 6))
    ax = sns.barplot(x=top_pkgs.values, y=top_pkgs.index, palette="Reds_r")
    plt.title("Top 10 Paquetes con más Vulnerabilidades", fontsize=14)
    plt.xlabel("Cantidad de Vulnerabilidades")
    plt.ylabel("Paquete")
    for p in ax.patches:
        ax.annotate(f'{int(p.get_width())}', (p.get_width(), p.get_y() + p.get_height() / 2.),
                    ha='left', va='center', fontsize=10, color='black', xytext=(5, 0),
                    textcoords='offset points')
    plt.tight_layout()
    plt.show()
else:
    print("Sin datos de paquetes.")

---
## 2. Análisis Estático de Código (Semgrep / SAST)

Semgrep analiza el código fuente directamente (no las dependencias) buscando patrones de vulnerabilidad como inyección SQL, XSS, hardcoded secrets, etc.
Su JSON de salida tiene la estructura:
```
{ "results": [ { "check_id": "...", "path": "...", "start": {"line": ...}, "extra": {"severity": "...", "message": "..."} } ] }
```

**Carga y parseo de resultados Semgrep:** Se iteran los JSON en `SAST_DIR`, se extraen los hallazgos del array `results` y se mapean a un DataFrame con rule ID, severidad, mensaje, archivo y línea.

In [ ]:
all_sast = []

for filename in os.listdir(SAST_DIR):
    if filename.endswith(".json"):
        filepath = os.path.join(SAST_DIR, filename)
        with open(filepath, "r") as f:
            data = json.load(f)
            repo_name = filename.replace("_sast.json", "")

            results = data.get("results", [])
            for finding in results:
                extra = finding.get("extra", {})
                all_sast.append({
                    "Repository": repo_name,
                    "Rule ID": extra.get("metadata", {}).get("id", finding.get("check_id", "")),
                    "Severity": extra.get("severity", "Unknown"),
                    "Message": extra.get("message", ""),
                    "File": finding.get("path", ""),
                    "Line": finding.get("start", {}).get("line", "")
                })

df_sast = pd.DataFrame(all_sast)
print(f"Total de hallazgos SAST: {len(df_sast)}")
if not df_sast.empty:
    display(df_sast.head())

**Distribución SAST por severidad:** Equivalente al gráfico de Grype pero para hallazgos de código fuente.

In [ ]:
if not df_sast.empty:
    plt.figure(figsize=(10, 6))
    ax = sns.countplot(
        data=df_sast,
        x="Severity",
        hue="Severity",
        order=[s for s in SEVERITY_ORDER if s in df_sast["Severity"].values],
        palette="Blues_r",
        legend=False
    )
    plt.title("Distribución de Hallazgos SAST por Severidad (Semgrep)", fontsize=14)
    plt.ylabel("Cantidad")
    plt.xlabel("Nivel de Severidad")

    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=10, color='black', xytext=(0, 5),
                    textcoords='offset points')
    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron hallazgos SAST.")

**Hallazgos SAST por repositorio:** Muestra qué repositorios tienen más problemas de seguridad en su código fuente.

In [ ]:
if not df_sast.empty:
    plt.figure(figsize=(12, 6))
    ax = sns.countplot(
        data=df_sast,
        x="Repository",
        hue="Severity",
        hue_order=[s for s in SEVERITY_ORDER if s in df_sast["Severity"].values],
        palette="Blues_r"
    )
    plt.title("Hallazgos SAST por Repositorio y Severidad", fontsize=14)
    plt.ylabel("Cantidad")
    plt.xlabel("Repositorio")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("Sin datos SAST por repositorio.")

**Top 10 reglas más frecuentes:** Identifica qué patrones de vulnerabilidad se repiten más. Esto ayuda a priorizar correcciones (ej: si una regla de SQL injection aparece 50 veces, hay un patrón sistemático).

In [ ]:
if not df_sast.empty:
    top_rules = df_sast["Rule ID"].value_counts().head(10)
    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x=top_rules.values, y=[str(r)[:50] for r in top_rules.index], palette="Blues_r")
    plt.title("Top 10 Reglas Semgrep más Frecuentes", fontsize=14)
    plt.xlabel("Cantidad de Hallazgos")
    plt.ylabel("Rule ID")
    for p in ax.patches:
        ax.annotate(f'{int(p.get_width())}', (p.get_width(), p.get_y() + p.get_height() / 2.),
                    ha='left', va='center', fontsize=10, color='black', xytext=(5, 0),
                    textcoords='offset points')
    plt.tight_layout()
    plt.show()
else:
    print("Sin datos de reglas.")

---
## 3. Análisis Combinado: Grype + Semgrep

En esta sección se cruzan los resultados de ambas herramientas para obtener una visión completa del estado de seguridad: dependencias vulnerables (Grype) + código inseguro (Semgrep).

**Tabla resumen:** Totales de hallazgos por fuente de análisis.

In [ ]:
resumen = pd.DataFrame({
    "Vulns Dependencias (Grype)": [len(df_vulns)],
    "Hallazgos SAST (Semgrep)": [len(df_sast)],
    "Total": [len(df_vulns) + len(df_sast)]
})
display(resumen)

**Comparativa de severidad:** Gráfico lado a lado que muestra si los problemas de seguridad provienen más de las dependencias o del código propio en cada nivel de severidad.

In [ ]:
if not df_vulns.empty or not df_sast.empty:
    vulns_tagged = df_vulns[["Repository", "Severity"]].copy() if not df_vulns.empty else pd.DataFrame(columns=["Repository", "Severity"])
    sast_tagged = df_sast[["Repository", "Severity"]].copy() if not df_sast.empty else pd.DataFrame(columns=["Repository", "Severity"])

    if not vulns_tagged.empty:
        vulns_tagged["Source"] = "Grype (Dependencias)"
    if not sast_tagged.empty:
        sast_tagged["Source"] = "Semgrep (SAST)"

    df_combined = pd.concat([vulns_tagged, sast_tagged], ignore_index=True)

    if not df_combined.empty:
        plt.figure(figsize=(12, 6))
        ax = sns.countplot(
            data=df_combined,
            x="Severity",
            hue="Source",
            order=[s for s in SEVERITY_ORDER if s in df_combined["Severity"].values],
            palette={"Grype (Dependencias)": "#d73027", "Semgrep (SAST)": "#4575b4"}
        )
        plt.title("Comparativa de Severidad: Dependencias vs Código Fuente", fontsize=14)
        plt.ylabel("Cantidad")
        plt.xlabel("Nivel de Severidad")
        plt.tight_layout()
        plt.show()
    else:
        print("Sin datos combinados.")
else:
    print("No hay datos en ninguna de las dos fuentes.")

**Hallazgos totales por repositorio:** Gráfico apilado que muestra la contribución de cada herramienta al total de hallazgos por repo. Los repos con más barras son los de mayor riesgo.

In [ ]:
if not df_vulns.empty or not df_sast.empty:
    vulns_per_repo = df_vulns.groupby("Repository").size().rename("Grype") if not df_vulns.empty else pd.Series(dtype=int)
    sast_per_repo = df_sast.groupby("Repository").size().rename("Semgrep") if not df_sast.empty else pd.Series(dtype=int)

    df_repo_total = pd.concat([vulns_per_repo, sast_per_repo], axis=1).fillna(0).astype(int)
    df_repo_total["Total"] = df_repo_total["Grype"] + df_repo_total["Semgrep"]
    df_repo_total = df_repo_total.sort_values("Total", ascending=False)

    plt.figure(figsize=(12, 6))
    ax = df_repo_total[["Grype", "Semgrep"]].plot(kind="bar", stacked=True, ax=plt.gca(),
                                                   color=["#d73027", "#4575b4"], width=0.6)
    plt.title("Total de Hallazgos por Repositorio (Grype + Semgrep)", fontsize=14)
    plt.ylabel("Cantidad")
    plt.xlabel("Repositorio")
    plt.xticks(rotation=45, ha='right')

    for p in ax.patches:
        if p.get_height() > 0:
            ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_y() + p.get_height()),
                        ha='center', va='bottom', fontsize=9, color='black', xytext=(0, 3),
                        textcoords='offset points')
    plt.tight_layout()
    plt.show()
else:
    print("Sin datos por repositorio.")

---
## 4. Análisis de SBOMs (Dependencias Totales)

Los SBOMs (Software Bill of Materials) generados por Syft contienen el inventario completo de dependencias de cada repositorio.
Aquí se analiza la cantidad de dependencias y los lenguajes detectados por repo.

**Carga y estadísticas de SBOMs:** Se extrae el array `artifacts` de cada SBOM JSON, se cuentan las dependencias totales y se recopilan los lenguajes detectados.

In [ ]:
sbom_stats = []

for filename in os.listdir(SBOMS_DIR):
    if filename.endswith(".json"):
        filepath = os.path.join(SBOMS_DIR, filename)
        with open(filepath, "r") as f:
            data = json.load(f)
            repo_name = filename.replace("_sbom.json", "")
            artifacts = data.get("artifacts", [])

            languages = set()
            for art in artifacts:
                lang = art.get("language", "")
                if lang:
                    languages.add(lang)

            sbom_stats.append({
                "Repository": repo_name,
                "Total Dependencies": len(artifacts),
                "Languages": ", ".join(sorted(languages)) if languages else "N/A"
            })

df_sboms = pd.DataFrame(sbom_stats)
if not df_sboms.empty:
    print(f"Repositorios con SBOM: {len(df_sboms)}")
    print(f"Total de dependencias escaneadas: {df_sboms['Total Dependencies'].sum()}")
    display(df_sboms.sort_values("Total Dependencies", ascending=False))
else:
    print("No se encontraron archivos SBOM.")

**Dependencias por repositorio:** Gráfico de barras que muestra qué repositorios tienen más dependencias. Más dependencias = mayor superficie de ataque potencial.

In [ ]:
if not df_sboms.empty:
    plt.figure(figsize=(10, 6))
    ax = sns.barplot(data=df_sboms.sort_values("Total Dependencies", ascending=False),
                     x="Repository", y="Total Dependencies", palette="Greens_r")
    plt.title("Cantidad de Dependencias por Repositorio (SBOM)", fontsize=14)
    plt.ylabel("Número de Dependencias")
    plt.xlabel("Repositorio")
    plt.xticks(rotation=45, ha='right')

    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=10, color='black', xytext=(0, 5),
                    textcoords='offset points')
    plt.tight_layout()
    plt.show()
else:
    print("Sin datos de SBOM para graficar.")